# Week 13 Optional: Advanced DeepEval Metrics

This optional notebook goes deeper into DeepEval's evaluation capabilities.
You'll learn to evaluate Knowledge Base responses for faithfulness and
relevancy, and build custom evaluation criteria.

**Prerequisites**: Completed Week 13 main notebook (Bedrock setup, KB access)

**Platform**: AWS SageMaker (same as main notebook)

In [ ]:
# =============================================================================
# SETUP (same as main notebook)
# =============================================================================

!pip install -q deepeval aiobotocore

import boto3
import json
import os
import time
import re
import numpy as np
import pandas as pd

from deepeval.models import AmazonBedrockModel
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
KB_ID = os.environ.get("BEDROCK_KB_ID", "QWP1VUKMFS")

bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)

# Reuse query_knowledge_base from main notebook
def query_knowledge_base(question, kb_id=KB_ID, model_id='anthropic.claude-sonnet-4-5-20250929-v1:0'):
    model_arn = f'arn:aws:bedrock:{AWS_REGION}::foundation-model/{model_id}'
    response = bedrock_agent_runtime.retrieve_and_generate(
        input={'text': question},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': model_arn
            }
        }
    )
    answer = response['output']['text']
    citations = []
    for citation in response.get('citations', []):
        for ref in citation.get('retrievedReferences', []):
            source = ref.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
            snippet = ref.get('content', {}).get('text', '')[:150]
            citations.append({'source': source, 'snippet': snippet})
    return answer, citations

# Set up the judge model
judge_model = AmazonBedrockModel(
    model_id='anthropic.claude-sonnet-4-5-20250929-v1:0',
    region=AWS_REGION,
)

print("✅ Setup complete!")

# Section 1: Answer Relevancy

**Answer Relevancy** measures whether the LLM's response actually answers
the question that was asked. A high score means the answer is on-topic;
a low score means the model went off on a tangent.

In [ ]:
# =============================================================================
# Answer Relevancy — Is the response relevant to the question?
# =============================================================================

relevancy_metric = AnswerRelevancyMetric(model=judge_model, threshold=0.7)

# Test on a KB response
question = "What triggers a fraud alert for international wire transfers?"
kb_answer, citations = query_knowledge_base(question)

test_case = LLMTestCase(
    input=question,
    actual_output=kb_answer,
    retrieval_context=[c['snippet'] for c in citations]
)

relevancy_metric.measure(test_case)
print(f"Question: {question}")
print(f"Answer: {kb_answer[:200]}...")
print(f"Relevancy Score: {relevancy_metric.score:.2f}")
print(f"Reason: {relevancy_metric.reason}")

# Section 2: Faithfulness

**Faithfulness** measures whether the LLM stayed within the retrieved context.
Low faithfulness means the model is adding information not in your documents —
a form of hallucination that's particularly dangerous in regulated industries.

In [ ]:
# =============================================================================
# Faithfulness — Does the answer stick to the source documents?
# =============================================================================

faithfulness_metric = FaithfulnessMetric(model=judge_model, threshold=0.7)

test_case = LLMTestCase(
    input=question,
    actual_output=kb_answer,
    retrieval_context=[c['snippet'] for c in citations]
)

faithfulness_metric.measure(test_case)
print(f"Faithfulness Score: {faithfulness_metric.score:.2f}")
print(f"Reason: {faithfulness_metric.reason}")
print(f"\n💡 Faithfulness measures whether the LLM stayed within the retrieved context.")
print(f"   Low faithfulness = the model is adding information not in your documents.")

# Section 3: Custom G-Eval Criteria

G-Eval lets you define **custom evaluation criteria**. Instead of using
pre-built metrics, you describe what you want to measure in plain English
and the judge LLM evaluates accordingly.

In [ ]:
# =============================================================================
# Custom G-Eval — Define Your Own Evaluation Criteria
# =============================================================================

# Custom metric: Does the classification include a risk score?
risk_score_metric = GEval(
    name="Risk Score Inclusion",
    criteria="Evaluate whether the LLM output includes a risk assessment or confidence level for the fraud classification.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

# Get a detailed response from Bedrock
detail_prompt = (
    "Analyze this transaction for fraud risk. Provide: classification (fraud/legitimate), "
    "risk score (1-10), and reasoning.\n\n"
    "Transaction: \"Customer reports unauthorized wire transfer of $4,500 to unknown overseas "
    "account. No prior international transaction history. Transfer initiated at 3:47 AM local time.\""
)

response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[{'role': 'user', 'content': [{'text': detail_prompt}]}],
    inferenceConfig={'maxTokens': 200, 'temperature': 0.0}
)
detailed_output = response['output']['message']['content'][0]['text']

test_case = LLMTestCase(input=detail_prompt, actual_output=detailed_output)
risk_score_metric.measure(test_case)
print(f"Output: {detailed_output[:200]}...")
print(f"Risk Score Inclusion: {risk_score_metric.score:.2f}")
print(f"Reason: {risk_score_metric.reason}")

## Lab: Build an Evaluation Pipeline

### Your Task

Build a function that evaluates multiple KB queries across all three DeepEval
metrics (Relevancy, Faithfulness, G-Eval correctness). Create a summary report
identifying which queries produce the best and worst results.

### Steps

1. Define 5 different fraud-related queries
2. For each query, get the KB answer and citations
3. Evaluate each with AnswerRelevancy, Faithfulness, and your custom G-Eval
4. Create a summary DataFrame with scores
5. Identify the query with the lowest faithfulness — why might it score low?

### Homework Extension

Run the pipeline on 10+ queries covering edge cases. Which types of questions
does the KB handle best? Where does faithfulness break down?

In [ ]:
# =============================================================================
# SOLUTION: LAB — BUILD AN EVALUATION PIPELINE
# =============================================================================

# Define 5 fraud-related queries
eval_queries = [
    "What triggers a fraud alert for international wire transfers?",
    "What are the ATM withdrawal limits for debit cards?",
    "How should card-not-present transactions be verified?",
    "What is the procedure for freezing an account after suspected fraud?",
    "What red flags indicate account takeover fraud?",
]

# Build evaluation function
def evaluate_kb_query(query, relevancy_metric, faithfulness_metric):
    """Evaluate a single KB query across multiple metrics."""
    kb_answer, citations = query_knowledge_base(query)
    retrieval_context = [c['snippet'] for c in citations]

    test_case = LLMTestCase(
        input=query,
        actual_output=kb_answer,
        retrieval_context=retrieval_context
    )

    relevancy_metric.measure(test_case)
    faithfulness_metric.measure(test_case)

    return {
        'query': query,
        'answer_length': len(kb_answer),
        'num_citations': len(citations),
        'relevancy': relevancy_metric.score,
        'relevancy_reason': relevancy_metric.reason,
        'faithfulness': faithfulness_metric.score,
        'faithfulness_reason': faithfulness_metric.reason,
    }

# Run pipeline on all queries
pipeline_results = []
for i, query in enumerate(eval_queries, 1):
    print(f"Evaluating query {i}/{len(eval_queries)}: {query[:50]}...")
    result = evaluate_kb_query(query, relevancy_metric, faithfulness_metric)
    pipeline_results.append(result)
    print(f"  Relevancy: {result['relevancy']:.2f}, Faithfulness: {result['faithfulness']:.2f}")

# Create summary DataFrame
eval_df = pd.DataFrame(pipeline_results)

# Display results
print(f"\n{'='*60}")
print(f"Evaluated {len(eval_df)} queries")
print(f"Avg Relevancy:    {eval_df['relevancy'].mean():.2f}")
print(f"Avg Faithfulness: {eval_df['faithfulness'].mean():.2f}")

best = eval_df.loc[eval_df['faithfulness'].idxmax()]
worst = eval_df.loc[eval_df['faithfulness'].idxmin()]
print(f"\nHighest faithfulness: {best['query'][:50]}... ({best['faithfulness']:.2f})")
print(f"Lowest faithfulness:  {worst['query'][:50]}... ({worst['faithfulness']:.2f})")
print(f"\n🎉 Evaluation pipeline complete!")

# Summary

You've explored three powerful DeepEval metrics:

| Metric | What It Measures | When to Use |
|--------|-----------------|-------------|
| **Answer Relevancy** | Is the answer on-topic? | Any LLM output |
| **Faithfulness** | Does the answer stick to sources? | RAG / KB responses |
| **G-Eval (custom)** | Whatever you define | Domain-specific criteria |

These metrics form the foundation of a production evaluation pipeline.
In Week 14, you could evaluate your fine-tuned model against these same
benchmarks to measure improvement.

## Resources

- [DeepEval Metrics Guide](https://deepeval.com/docs/metrics-introduction)
- [G-Eval Paper](https://arxiv.org/abs/2303.16634)
- [Faithfulness in RAG Systems](https://deepeval.com/docs/metrics-faithfulness)